# testing new memes

In [21]:
import requests
from IPython.display import display, Image, HTML
import os
from dotenv import load_dotenv
USERNAME = os.getenv('IMGFLIP_USERNAME')
PASSWORD = os.getenv('IMGFLIP_PASSWORD')
URL = "https://api.imgflip.com/caption_image"

In [22]:
def test_meme(input_dict):
    # append creds
    input_dict['username'] = USERNAME
    input_dict['password'] = PASSWORD
    

    # post and raise if issue
    response = requests.post(URL, data=input_dict)
    response.raise_for_status()

    # Parse the JSON response
    result = response.json()
    image_url = result['data']['url']

    # display in notebook
    display(Image(url=image_url))

In [23]:
test_meme(
{
    "template_id": "27813981",
    "text0": "When they open Powerpoint and you see slide '1 out of 243'",
    "text1": ""
  }




)

In [24]:
# from 10 to 27
import json
with open('../apps/gradio/memes20250304.json','r') as file:
    memes = json.load(file)

len(memes)

27

# we have more memes, so what? Let's try evaluating
- I will attempt to create a harness to see if memes improve scores
- We previously evaluated against 29 prompts using 4 binary criteria - relevant, surprising, creative, like_it - and one comment text entry
- Some researchers have attempted humour, creativity, shareability - https://arxiv.org/pdf/2501.11433
- Regardless excel is awful - so let's try jupyter notebook and maybe save to a local database

In [25]:
# get prompts
import pickle
with open('../data/evals/prompts.pkl', 'rb') as file:
    prompts = pickle.load(file)

len(prompts)

29

In [32]:
# 1. SETUP AND CONFIGURATION
import os
import json
import random
import pickle
import requests
import sqlite3
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, Image, HTML
import ipywidgets as widgets
import google.generativeai as genai

# Load environment variables
load_dotenv()

# load files
app_path = '../apps/gradio/'
with open(app_path+'system_prompt.txt', 'r') as f:
    system_prompt = f.read()

with open(app_path+'memes20250128.pkl', 'rb') as f:
    meme_context = pickle.load(f)


# Configure Gemini
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
)

# Create or connect to SQLite database
def setup_database():
    conn = sqlite3.connect('../data/evals/meme_evaluations.db')
    cursor = conn.cursor()
    
    # Create table if it doesn't exist
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS meme_evaluations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        test_title TEXT NOT NULL,
        timestamp TEXT NOT NULL,
        prompt TEXT NOT NULL,
        gemini_json TEXT,
        imgflip_url TEXT,
        is_humorous INTEGER,
        is_creative INTEGER,
        is_shareable INTEGER,
        text_comments TEXT
    )
    ''')
    
    conn.commit()
    return conn, cursor

# Initialize database
conn, cursor = setup_database()
print("Database initialized successfully.")

# Helper function to clean JSON responses
def clean_response(response: str) -> str:
    """
    Cleans AI response to ensure valid JSON formatting.
    Handles edge cases with 'json' prefix and smart quote issues.
    """
    # Remove 'json' prefix and newlines at start
    cleaned = response.lstrip('json\n')
    
    # Remove markdown code blocks and backticks
    cleaned = cleaned.strip('`').replace('```json', '').replace('```', '')
    
    # Replace various quote types with standard double quotes
    cleaned = cleaned.replace('"', '"').replace('"', '"').replace("'", '"')
    
    # Handle escaped quotes inside text strings
    cleaned = cleaned.replace('\\"', '"')
    
    # Remove any trailing commas before closing braces
    cleaned = cleaned.replace(',}', '}')
    cleaned = cleaned.replace(',]', ']')
    
    return cleaned.strip()

Database initialized successfully.


# Pivoting to a gradio app